In [2]:
URI = "bolt://localhost:7687"
USER = "neo4j"
PASSWORD = "neo4jiscool"
DATABASE = "neo4j"

In [3]:
driver = Neo4jGraphDriver(URI, USER, PASSWORD, DATABASE)

# graph = driver.pull_graph(verbose=False)
stats = driver.summarize_database(return_dict=True)


Graph Summary

       36458 Nodes
           0 Relationships

Node Labels (Count)

       36222 Folder
         115 AudioRecording
           2 Artist
           4 Album
         115 Name

Relationship Types (Count)




In [4]:
import pickle

# load graph object from file
graph = pickle.load(open('neo4j_db.pickle', 'rb'))

In [5]:
from random import randint

def summarize_node(graph, uid):
    node = graph.nodes(data=True)[uid]
    print()
    print(f"       UID: {uid}")
    print(f"    Labels: {node['labels']}")
    print( "Properties: {")
    for k, v in node.items():
        if k != 'labels':
            print(f"       {k:>13}: {v:<}")
    print( "            }")
    print()

def random_uid(graph):
    return str(list(graph.nodes().keys())[randint(0,len(graph.nodes()))])

summarize_node(graph, random_uid(graph))


       UID: 4:7abf331b-17a7-43e1-8474-9ab67d56a5da:42044
    Labels: ['Folder']
Properties: {
                  id: 4:7abf331b-17a7-43e1-8474-9ab67d56a5da:42044
            filepath: /home/patch/anaconda3/pkgs/qt-main-5.15.2-h327a75a_7/share/qt/3rd_party_licenses/qtwebengine/src/3rdparty/chromium/third_party/devtools-frontend/src/node_modules/mkdirp
                uuid: be928d4b-95d2-469d-b06e-3e7244c63f79
            }



In [6]:
max_iter = 3
count=0
for uid in list(graph.nodes())[:max_iter]:
    summarize_node(graph, uid)


       UID: 4:7abf331b-17a7-43e1-8474-9ab67d56a5da:0
    Labels: ['Folder']
Properties: {
                  id: 4:7abf331b-17a7-43e1-8474-9ab67d56a5da:115
            filepath: /home/patch/Music/Untitled-2023-08-01-14-09-20/interchange/Untitled-2023-08-01-14-09-20
                uuid: 0716591c-5b1d-456a-b875-3d244b52ca47
            }


       UID: 4:7abf331b-17a7-43e1-8474-9ab67d56a5da:1
    Labels: ['Folder']
Properties: {
                  id: 4:7abf331b-17a7-43e1-8474-9ab67d56a5da:116
            filepath: /home/patch/Music/Untitled-2023-08-01-14-09-20/interchange/Untitled-2023-08-01-14-09-20/midifiles
                uuid: 90a4a005-c4e1-4127-bfa1-c110fa00b464
            }


       UID: 4:7abf331b-17a7-43e1-8474-9ab67d56a5da:2
    Labels: ['Folder']
Properties: {
                  id: 4:7abf331b-17a7-43e1-8474-9ab67d56a5da:117
            filepath: /home/patch/Music/Untitled-2023-08-01-14-09-20/interchange/Untitled-2023-08-01-14-09-20/audiofiles
                uuid: da88555c-be

In [7]:
# Compare graphs
diff = driver.compare_graph_with_database(graph)
print("Graph differences:", diff)

Graph differences: {'missing_nodes': 6806, 'extra_nodes': 49, 'missing_edges': 43245, 'extra_edges': 0}


Delete some relationships directly in the database... then

In [8]:
# Compare graphs
diff = driver.compare_graph_with_database(graph, detailed=False)
print("Graph differences:", diff)

Graph differences: {'missing_nodes': 6806, 'extra_nodes': 49, 'missing_edges': 43245, 'extra_edges': 0}


Now, pull the current graph...

In [9]:
graph2 = driver.pull_graph(verbose=True)

Graph pulled with 36458 nodes and 0 edges.


... and push the checkpointed graph, replacing the previous one... 

In [10]:
driver.push_graph(graph, clear_existing=True)

Uploading Relationships:   0%|                                                    | 0/44 [00:00<?, ?it/s]


CypherSyntaxError: {code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input '{': expected a node label/relationship type name, '$', '%' or '(' (line 4, column 22 (offset: 116))
"        MERGE (a)-[r:{rel.rel_type}]->(b)"
                      ^}

In [17]:
import pickle

# save graph object to file
pickle.dump(graph, open('neo4j_db.pickle', 'wb'))



In [24]:
graph3 = driver.pull_graph(verbose=True)

Graph pulled with 43215 nodes and 43245 edges.


In [1]:
from neo4j import GraphDatabase
import networkx as nx
import uuid
from tqdm import tqdm

class Neo4jGraphDriver:
    def __init__(self, uri, user, password, database="neo4j"):
        self.uri = uri
        self.user = user
        self.password = password
        self.database = database
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        
        if not self.database_exists():
            self.initialize_database()

    def close(self):
        self.driver.close()
    
    def database_exists(self):
        """Check if the specified database exists."""
        with self.driver.session() as session:
            result = session.run("SHOW DATABASES")
            return any(record["name"] == self.database for record in result)
    
    def initialize_database(self):
        """Initialize a new database if it doesn't exist."""
        with self.driver.session() as session:
            session.run(f"CREATE DATABASE {self.database}")
    
    def pull_graph(self, verbose=True):
        """Pulls the entire Neo4j database into a NetworkX MultiDiGraph."""
        G = nx.MultiDiGraph()
        
        with self.driver.session(database=self.database) as session:
            # Fetch nodes
            nodes_query = "MATCH (n) RETURN elementId(n) AS id, labels(n) AS labels, properties(n) AS props"
            result = session.run(nodes_query)
            for record in result:
                G.add_node(record['id'], labels=record['labels'], **record['props'])
            
            # Fetch relationships
            rels_query = "MATCH (a)-[r]->(b) RETURN elementId(a) AS src, elementId(b) AS tgt, type(r) AS type, properties(r) AS props"
            result = session.run(rels_query)
            for record in result:
                G.add_edge(record['src'], record['tgt'], key=record['type'], type=record['type'], **record['props'])
                
        if verbose:
            print(f"Graph pulled with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")
            
        return G
    
    def push_graph(self, G, clear_existing=False, batch_size=1000):
        """Pushes a NetworkX MultiDiGraph into Neo4j efficiently with batched inserts and progress bar."""
        with self.driver.session(database=self.database) as session:
            if clear_existing:
                session.run("MATCH (n) DETACH DELETE n")
            
            # Add nodes in batches with progress bar
            nodes = list(G.nodes(data=True))
            for i in tqdm(range(0, len(nodes), batch_size), desc="Uploading Nodes"):
                batch = nodes[i:i + batch_size]
                self._push_node_batch(session, batch)
            
            # Add relationships in batches with progress bar
            relationships = list(G.edges(keys=True, data=True))
            for i in tqdm(range(0, len(relationships), batch_size), desc="Uploading Relationships"):
                batch = relationships[i:i + batch_size]
                self._push_relationship_batch(session, batch, G)
    
    def _push_node_batch(self, session, nodes):
        """Helper function to push a batch of nodes."""
        query = """
        UNWIND $batch AS node
        MERGE (n {uuid: node.uuid})
        SET n += node.props
        FOREACH (label IN node.labels | SET n:label)
        """
        batch_data = [
            {
                "uuid": node_data.get('uuid', str(uuid.uuid4())),
                "props": {k: v for k, v in node_data.items() if k not in ['labels', 'uuid']},
                "labels": node_data.get('labels', ['Node'])
            }
            for _, node_data in nodes
        ]
        session.run(query, batch=batch_data)
    
    def _push_relationship_batch(self, session, relationships, G):
        """Helper function to push a batch of relationships."""
        query = """
        UNWIND $batch as rel
        MATCH (a {uuid: rel.src_uuid}), (b {uuid: rel.tgt_uuid})
        MERGE (a)-[r:{rel.rel_type}]->(b)
        SET r += rel.props
        """
        batch_data = [
            {
                "src_uuid": G.nodes[src]['uuid'],
                "tgt_uuid": G.nodes[tgt]['uuid'],
                "rel_type": edge_data.get('type', 'RELATED_TO'),
                "props": {k: v for k, v in edge_data.items() if k != 'type'}
            }
            for src, tgt, key, edge_data in relationships
        ]
        session.run(query, batch=batch_data)
    
    def compare_graph_with_database(self, G, detailed=False):
        """Compare NetworkX graph with the database to check for differences."""
        db_graph = self.pull_graph(verbose=False)
        
        missing_nodes = set(G.nodes()) - set(db_graph.nodes())
        extra_nodes = set(db_graph.nodes()) - set(G.nodes())
        
        missing_edges = set(G.edges()) - set(db_graph.edges())
        extra_edges = set(db_graph.edges()) - set(G.edges())
        
        summary_diff = {
            "missing_nodes": len(missing_nodes),
            "extra_nodes": len(extra_nodes),
            "missing_edges": len(missing_edges),
            "extra_edges": len(extra_edges),
        }
        
        if not detailed:
            return summary_diff
        
        return {
            "missing_nodes": list(missing_nodes),
            "extra_nodes": list(extra_nodes),
            "missing_edges": list(missing_edges),
            "extra_edges": list(extra_edges),
        }

    def get_database_stats(self):
        """Retrieve statistics on nodes, relationships, and properties."""
        with self.driver.session(database=self.database) as session:
            node_count = session.run("MATCH (n) RETURN count(n) AS count").single()["count"]
            rel_count = session.run("MATCH ()-[r]->() RETURN count(r) AS count").single()["count"]
            
            node_labels = session.run("MATCH (n) RETURN labels(n) AS label, count(n) AS count")
            node_labels = {record["label"][0]: record["count"] for record in node_labels if record["label"]}
            
            rel_types = session.run("MATCH ()-[r]->() RETURN type(r) AS type, count(r) AS count")
            rel_types = {record["type"]: record["count"] for record in rel_types}
            
            return {
                "total_nodes": node_count,
                "total_relationships": rel_count,
                "nodes_by_label": node_labels,
                "relationships_by_type": rel_types,
            }
    
    def summarize_database(self, return_dict=False):
        stats = self.get_database_stats()
        print()
        print(f"Graph Summary")
        print()
        print(f"  {int(stats['total_nodes']):10d} Nodes")
        print(f"  {int(stats['total_relationships']):10d} Relationships")
        print()
        print("Node Labels (Count)")
        print()
        for _label, _count in stats['nodes_by_label'].items():
            print(f"  {int(_count):10d} {_label}")
        print()
        print("Relationship Types (Count)")
        print()
        for _type, _count in stats['relationships_by_type'].items():
            print(f"  {int(_count):10d} {_type}")
        print()
        if return_dict:
            return stats
